In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
import time

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

batch_size = 128
lr = 0.001
epochs = 5
epsilon = 8/255
alpha = 2/255
pgd_steps = 7
beta = 6.0

transform = transforms.Compose([transforms.ToTensor()])

trainset = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=transform)
testset = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=transform)

trainloader = torch.utils.data.DataLoader(trainset, batch_size=batch_size, shuffle=True)
testloader = torch.utils.data.DataLoader(testset, batch_size=batch_size, shuffle=False)

class CNN(nn.Module):
    def __init__(self):
        super(CNN, self).__init__()
        self.net = nn.Sequential(
            nn.Conv2d(3,32,3,padding=1), nn.ReLU(),
            nn.Conv2d(32,64,3,padding=1), nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(64,128,3,padding=1), nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Flatten(),
            nn.Linear(8*8*128,256), nn.ReLU(),
            nn.Linear(256,10)
        )
    def forward(self,x):
        return self.net(x)

def clamp(X):
    return torch.clamp(X,0,1)

def fgsm(model, X, y):
    model.eval()
    X_adv = X.clone().detach()
    X_adv = X_adv + torch.zeros_like(X_adv).uniform_(-epsilon, epsilon)
    X_adv = clamp(X_adv)

    X_adv.requires_grad_(True)
    output = model(X_adv)
    loss = nn.CrossEntropyLoss()(output, y)

    model.zero_grad()
    loss.backward()
    X_adv = X_adv + epsilon * X_adv.grad.sign()

    return clamp(X_adv.detach())

def pgd(model, X, y):
    model.eval()
    X_adv = X.clone().detach()
    X_adv = X_adv + torch.zeros_like(X_adv).uniform_(-epsilon, epsilon)
    X_adv = clamp(X_adv)

    for _ in range(pgd_steps):
        X_adv.requires_grad_(True)
        loss = nn.CrossEntropyLoss()(model(X_adv), y)

        model.zero_grad()
        loss.backward()

        X_adv = X_adv + alpha * X_adv.grad.sign()
        X_adv = torch.min(torch.max(X_adv, X - epsilon), X + epsilon)
        X_adv = clamp(X_adv.detach())

    return X_adv

def evaluate(model):
    model.eval()
    correct = 0
    total = 0
    adv_correct = 0
    with torch.no_grad():
        for X,y in testloader:
            X,y = X.to(device), y.to(device)
            out = model(X)
            _,pred = torch.max(out,1)
            correct += (pred==y).sum().item()
            total += y.size(0)

    for X,y in testloader:
        X,y = X.to(device), y.to(device)
        X_adv = pgd(model,X,y)
        out = model(X_adv)
        _,pred = torch.max(out,1)
        adv_correct += (pred==y).sum().item()

    return correct/total, adv_correct/total

def train_pgd():
    model = CNN().to(device)
    opt = optim.Adam(model.parameters(), lr=lr)
    start = time.time()
    for _ in range(epochs):
        model.train()
        for X,y in trainloader:
            X,y = X.to(device), y.to(device)
            X_adv = pgd(model,X,y)

            model.train()
            opt.zero_grad()
            loss = nn.CrossEntropyLoss()(model(X_adv),y)
            loss.backward()
            opt.step()
    t = time.time()-start
    acc,rob = evaluate(model)
    return t,acc,rob

def train_trades():
    model = CNN().to(device)
    opt = optim.Adam(model.parameters(), lr=lr)
    kl = nn.KLDivLoss(reduction='batchmean')
    start = time.time()
    for _ in range(epochs):
        model.train()
        for X,y in trainloader:
            X,y = X.to(device), y.to(device)

            model.eval()
            X_adv = X.clone().detach() + 0.001 * torch.randn_like(X)

            with torch.no_grad():
                clean_output = model(X).detach()
            clean_prob = nn.Softmax(dim=1)(clean_output)

            for _ in range(pgd_steps):
                X_adv.requires_grad_(True)
                adv_log_prob = nn.LogSoftmax(dim=1)(model(X_adv))
                loss_kl = kl(adv_log_prob, clean_prob)

                model.zero_grad()
                loss_kl.backward()

                X_adv = X_adv + alpha * X_adv.grad.sign()
                X_adv = torch.min(torch.max(X_adv, X - epsilon), X + epsilon)
                X_adv = clamp(X_adv.detach())

            model.train()
            opt.zero_grad()

            logits_clean = model(X)
            logits_adv = model(X_adv)
            loss_nat = nn.CrossEntropyLoss()(logits_clean, y)
            loss_rob = kl(nn.LogSoftmax(dim=1)(logits_adv), nn.Softmax(dim=1)(logits_clean))
            loss = loss_nat + beta * loss_rob

            loss.backward()
            opt.step()
    t = time.time()-start
    acc,rob = evaluate(model)
    return t,acc,rob

def train_mart():
    model = CNN().to(device)
    opt = optim.Adam(model.parameters(), lr=lr)
    start = time.time()
    for _ in range(epochs):
        model.train()
        for X,y in trainloader:
            X,y = X.to(device), y.to(device)
            X_adv = pgd(model,X,y)

            model.train()
            opt.zero_grad()
            out_adv = model(X_adv)
            out = model(X)
            loss = nn.CrossEntropyLoss()(out_adv,y)
            prob = torch.softmax(out,dim=1)
            true_prob = prob.gather(1,y.unsqueeze(1)).squeeze()
            loss += torch.mean((1-true_prob) * nn.CrossEntropyLoss(reduction='none')(out,y))
            loss.backward()
            opt.step()
    t = time.time()-start
    acc,rob = evaluate(model)
    return t,acc,rob

def train_fast():
    model = CNN().to(device)
    opt = optim.Adam(model.parameters(), lr=lr)
    start = time.time()
    for _ in range(epochs):
        model.train()
        for X,y in trainloader:
            X,y = X.to(device), y.to(device)
            X_adv = fgsm(model,X,y)

            model.train()
            opt.zero_grad()
            loss = nn.CrossEntropyLoss()(model(X_adv),y)
            loss.backward()
            opt.step()
    t = time.time()-start
    acc,rob = evaluate(model)
    return t,acc,rob

results = {}

t,acc,rob = train_pgd()
results['PGD-AT'] = (t,acc,rob)

t,acc,rob = train_trades()
results['TRADES'] = (t,acc,rob)

t,acc,rob = train_mart()
results['MART'] = (t,acc,rob)

t,acc,rob = train_fast()
results['FAST-AT'] = (t,acc,rob)

for k,v in results.items():
    print(k)
    print(f"Time: {v[0]:.2f} sec")
    print(f"Clean Acc: {v[1]*100:.2f}%")
    print(f"Robust Acc: {v[2]*100:.2f}%")
    print()